In [ ]:
### Run MOFA Model on the data generated in the previous script (02)

#############################################
# Prerequisites - Load Libraries

In [ ]:
source('MS1_Functions.r')

In [ ]:
### Informa about start of execution

popup_function_pos('03_Run_MOFA: Execution Started')

In [ ]:
source('MS0_Libraries.r')

In [ ]:
source('MS2_Plot_Config.r')

In [ ]:
#py_config() # - To check the configuration which python package will be used for MOFA

###############################################
# Preqrequisites Configurations & Parameters

In [ ]:
### Load the parameters that are set via the configuration files

In [ ]:
### Load configurations file
global_configs = read.csv('configurations/Data_Configs.csv', sep = ',')

In [ ]:
head(global_configs,2)

In [ ]:
data_path = global_configs$value[global_configs$parameter == 'data_path']

In [ ]:
data_path

In [ ]:
result_path = global_configs$value[global_configs$parameter == 'result_path']

In [ ]:
result_path

In [ ]:
## MOFA Model Configurations

In [ ]:
mofa_configs = read.csv( 'configurations/03_MOFA_Configs.csv', sep = ',')
mofa_configs = mofa_configs[mofa_configs$configuration_name != '',]

In [ ]:
head(mofa_configs,2)

In [ ]:
### Generate the result data directory if it does not exist yet
if(!file.exists(paste0(result_path, '03_results'))){
    dir.create(file.path(paste0(result_path, '03_results')))
    }

# Load Data 

## Prepared combined data

In [ ]:
### Load the data that was generated in the previous script using the name specified in the configuration file

In [ ]:
input_data = list()

In [ ]:
for(i in 1:nrow(mofa_configs)){
    path = paste0(result_path, '/02_results/02_Combined_Data_', mofa_configs$configuration_name[i] ,'_INTEGRATED.csv')
    
    if(file.exists(path)){
        data_long = read.csv(path)
        data_long$X = NULL
        print(path)
        print(file.info(path)$mtime)
        input_data[[i]]= data_long
        popup_function_pos('Prepared Data has been loaded')
        }
    else{
        popup_function_neg(paste0( 'Error: Data at' , result_path, '/02_results/', ' 02_Combined_Data_', mofa_configs$configuration_name[i] ,'_INTEGRATED.csv', ' could not be loaded. Make sure that all the previous steps have been executed successfully'))
     }   
        
    }

In [ ]:
length(unique(input_data[[1]]$variable))

In [ ]:
unique(input_data[[1]]$type)

In [ ]:
length(unique(input_data[[1]]$sample_id))

# Train MOFA Model

## Prepare data list

In [ ]:
### Adjust single-cell types to correspond to cell-types

In [ ]:
head(input_data[[1]],2)

In [ ]:
### Prepare data list for MOFA (adjust format of input data to be used as input for MOFA)

In [ ]:
input_data_list = list()

In [ ]:
data_list = list()

In [ ]:
input_data_list = lapply(input_data, function(x){

    for(i in unique(x$type)){
        samples = unique(x$sample_id) # necessary to have all samples in all dimensions
        data = x[x$type == i, ]

        data$type = NULL
        data$cell_type = NULL

        data = data %>% dcast(variable ~ sample_id, value  = "value")
        rownames(data) = data$variable
        colnames(data) = str_replace(colnames(data), 'value\\.', '')
        data$variable = NULL

        data[setdiff( samples, names(data))] = NA  # use all samples

        data = data[,order(colnames(data))]
        data = data[,colnames(data) %in% samples]

        data_list[[i]] = as.matrix(data)
        }
    
    return(data_list)
    })

In [ ]:
head(input_data_list[[1]][[1]],2)

In [ ]:
#str(input_data_list)

## Create MOFA object

In [ ]:
### Create a MOFA object to run the MOFA model on it

In [ ]:
names(input_data_list[[1]])

In [ ]:
mofa_object = lapply(input_data_list, function(x){
    MOFAobject = create_mofa(x)
    }
                     )

In [ ]:
### Plot the Data Overview showing the input used for the MOFA Model

In [ ]:
# Specific Text Descriptions:
xlabel = xlab('Samples') 
ylabel = ylab('View')

In [ ]:
# Sizes of the plot
width_par = 5
height_par =5

In [ ]:
options(repr.plot.width=30, repr.plot.height=10)

mofa_overview = lapply(mofa_object, function(x){
    mofa_overview = plot_data_overview(x)
    mofa_overview = mofa_overview + plot_config + theme(axis.text.y = element_text(hjust = 0, vjust = 0.5)) +
                xlabel + ylabel + theme(axis.text.x = element_blank())
    })

In [ ]:
#mofa_overview[[1]]

In [ ]:
# Extract data -type colors (used by the function to align and use those colors in the next plots)
type_colors = list()
for(i in 1:length(mofa_overview)){
    color_extraction =  ggplot_build(mofa_overview[[i]])
    type_colors[[i]] = unique(color_extraction$data[[1]]["fill"][,1])
    type_colors[[i]] = type_colors[[i]][!type_colors[[i]] == 'grey']
    }
    

In [ ]:
type_colors

In [ ]:
figure_name = "FIG03_Overview_MOFA_Input_"

In [ ]:

for(i in 1:length(mofa_overview)){
    pdf(paste0('figures/03_figures/', figure_name, mofa_configs$mofa_result[i],  '.pdf'), width =width_par, height =height_par)
    print(mofa_overview[[i]] )
    dev.off()
    }
popup_function_pos(paste0('Generated: figures/03_figures/ ', figure_name, mofa_configs$mofa_result[i],  '.pdf'))

## Set MOFA Training Options and run the Model Training

In [ ]:
### Define the MOFA parameters for training and run the model training with the set parameters
### Some parameters are handed over by thhe configuration file
### Others are currently assigned fixed below but can be modified

In [ ]:
model_result = list()

In [ ]:
for(i in 1:length(mofa_object)){
    
    ## Set other parameters of MOFA Model
    mefisto_opts = get_default_mefisto_options(mofa_object[[i]])
    
    ## Data Options
    data_opts = get_default_data_options(mofa_object[[i]])
    data_opts$scale_views = mofa_configs$scale_views[i] # decide whether to scale the data
    data_opts$use_float32 = FALSE
    print(data_opts)
    
    ## Model Options
    model_opts = get_default_model_options(mofa_object[[i]])
    model_opts$num_factors = mofa_configs$amount_of_factors[i] # define number of factors
    model_opts$spikeslab_weights = TRUE
    
    # model_opts$likelihoods['neutrophil'] = 'poisson' - example to modify distribution for one specific view
    #model_opts$likelihoods['cnvamp.filtered10'] = 'bernoulli'

    print(model_opts)
    
    ## Training Options
    train_opts  = get_default_training_options(mofa_object[[i]])
    train_opts$maxiter = 50000
    train_opts$verbose = TRUE
    train_opts$seed = 42
    train_opts$weight_views = mofa_configs$weighting_of_views[i]
    print(train_opts)
    
    ## Build and train the model
    MOFAobject = prepare_mofa(
      object = mofa_object[[i]],
      data_options = data_opts,
      model_options = model_opts,
      mefisto_options = mefisto_opts,
      training_options = train_opts #,
      #stochastic_options = stoch_options
    )
    
    model_name = paste0("03_MOFA_MODEL_", mofa_configs$mofa_result[i], '.hdf5')
    outfile = file.path( paste0(result_path, '/03_results/',  model_name) )
    print(outfile)
    MOFAobject.trained = run_mofa(MOFAobject, outfile, use_basilisk = FALSE)
    

    model_result[[i]] = MOFAobject.trained
    
    }
    

In [ ]:
str(mofa_object)

In [ ]:
for(i in 1:length(mofa_object)){
    
    ## Set other parameters of MOFA Model
    mefisto_opts = get_default_mefisto_options(mofa_object[[i]])
    
    ## Data Options
    data_opts = get_default_data_options(mofa_object[[i]])
    data_opts$scale_views = mofa_configs$scale_views[i] # decide whether to scale the data
    data_opts$use_float32 = FALSE
    print(data_opts)
    
    ## Model Options
    model_opts = get_default_model_options(mofa_object[[i]])
    model_opts$num_factors = mofa_configs$amount_of_factors[i] # define number of factors
    model_opts$spikeslab_weights = TRUE
    
    # model_opts$likelihoods['neutrophil'] = 'poisson' - example to modify distribution for one specific view
    #model_opts$likelihoods['cnvamp.filtered10'] = 'bernoulli'

    print(model_opts)
    
    ## Training Options
    train_opts  = get_default_training_options(mofa_object[[i]])
    train_opts$maxiter = 50000
    train_opts$verbose = TRUE
    train_opts$seed = 42
    train_opts$weight_views = mofa_configs$weighting_of_views[i]
    print(train_opts)
    
    ## Build and train the model
    MOFAobject = prepare_mofa(
      object = mofa_object[[i]],
      data_options = data_opts,
      model_options = model_opts,
      mefisto_options = mefisto_opts,
      training_options = train_opts #,
      #stochastic_options = stoch_options
    )
    
    model_name = paste0("03_MOFA_MODEL_", mofa_configs$mofa_result[i], '.hdf5')
    outfile = file.path( paste0(result_path, '/03_results/',  model_name) )
    print(outfile)
    MOFAobject.trained = run_mofa(MOFAobject, outfile, use_basilisk = FALSE)
    

    model_result[[i]] = MOFAobject.trained
    
    }
    

In [ ]:
str(model_result)

In [ ]:
#reticulate::py_config()

# Extract and prepare data for plots

In [ ]:
### Extract generated data for the model to use for later downstream analysis

## Extract Variance decomposition

In [ ]:
# Extract the total explained variance per view and factor

In [ ]:
model_result[[1]]@cache[["variance_explained"]]$r2_total  # per view

In [ ]:
rowMeans(model_result[[1]]@cache$variance_explained$r2_per_factor[[1]]) # per factor

In [ ]:
# Mean total variance explained

In [ ]:
mean(model_result[[1]]@cache$variance_explained$r2_total[[1]])

In [ ]:
# Save the explained variance

In [ ]:
for(i in 1:length(model_result)){
    write.csv(model_result[[i]]@cache$variance_explained$r2_per_factor[[1]], paste0(result_path, '/03_results/03_MOFA_Variance_Decomposition_',mofa_configs$mofa_result[i], '.csv'))
    }
popup_function_pos(paste0('Saved:', result_path, ' /03_results/ 03_MOFA_Variance_Decomposition_',mofa_configs$mofa_result[i], '.csv'))

## Extract factor and weight data

In [ ]:
#### Extract sample factors  values and save

In [ ]:
for(i in 1:length(model_result)){
    factors = get_factors(model_result[[i]], factors = "all")
    factors = factors$group1
    head(factors,2)
    
    factors = as.data.frame(factors)
    factors$sample_id = rownames(factors)
    
    # Save as csv
    write.csv(factors, paste0(result_path, '/03_results/03_Factor_Data_' , mofa_configs$mofa_result[i],  '.csv'), row.names = FALSE)
    }
popup_function_pos(paste0('Saved:', result_path, ' /03_results/ 03_Factor_Data_' , mofa_configs$mofa_result[i],  '.csv'))

In [ ]:
### Extract weight data (feature factor weights) and save

In [ ]:
for(j in 1:length(model_result)){
    weights = get_weights(model_result[[j]], views = "all", factors = "all")
    weight_data = data.frame()
    
    for (i in names(weights)){
        data = data.frame(weights[[i]])
        data$type = i
        weight_data = rbind(weight_data,data)
        }
    weight_data$variable_name = rownames(weight_data)
    
    # Save as csv
    write.csv(weight_data, paste0(result_path, '/03_results/03_Weight_Data_' ,mofa_configs$mofa_result[j], '.csv'), row.names = FALSE)
    }
popup_function_pos(paste0('Saved:', result_path, ' /03_results/ 03_Weight_Data_' ,mofa_configs$mofa_result[j], '.csv'))

# Diagnostic Result Plots

In [ ]:
### Make the explained variance plot to analyze the model result

## Plot explained variance overview

In [ ]:
## Prepare the data format

In [ ]:
explained_variance = lapply(model_result, function(x) {
    data = x@cache$variance_explained$r2_per_factor[[1]]
    data = melt(data)
    
    total_variance = data.frame( view = rownames(x@cache[["variance_explained"]]$r2_total$group1,2),
                             total_variance = x@cache[["variance_explained"]]$r2_total$group1)
    data = merge(data, total_variance, by.x = 'Var2', by.y = 'view')
    data$Var2 = as.character(data$Var2)
    data$Var2 = factor(data$Var2, levels = sort(unique(data$Var2)))
    data = data[order(data$Var2),]
    }
                            )

In [ ]:
#### Plot complete explained variance (Heatmap)

In [ ]:
var_decomp = lapply(explained_variance, function(x){
    ggplot() + 
        scale_fill_gradient(low="white", high="black") + 
        xlabel + 
        ylabel +
        plot_config + theme(axis.text.x = element_text(angle = 90), legend.position = 'right')+ 
        geom_tile(data = x, mapping = aes(Var1,  Var2, fill= value))
    })

In [ ]:
### Combine the plot with total variance barplot per dimension

In [ ]:
# Specific Text Descriptions:
xlabel = xlab('View') 
ylabel = ylab('Explained Variance')

In [ ]:
comp_variance = lapply(explained_variance, function(x){
    data = x
    plot_complete = unique(data[,c('Var2', 'total_variance')])
    comp_variance = ggplot(plot_complete, aes(x=Var2, y = total_variance, fill = Var2)) + 
                geom_bar(stat="identity") + coord_flip() + 
                xlabel + 
                ylabel +
                plot_config + scale_fill_manual(values = unlist(type_colors))  ## currently uses same coloring as MOFA oveview
    })

In [ ]:
#comp_variance[[1]]

In [ ]:
### Combine both visualization

In [ ]:
figure_name = "FIG03_Overview_Variance_Decomposition"

In [ ]:
# Sizes of the plot
width_par = 8.07
height_par = 4  # 2.8

In [ ]:
for(i in 1:length(explained_variance)){
    legend = get_legend(var_decomp[[i]])
    
    combination1 = ggarrange(var_decomp[[i]] + theme(legend.position = 'none'),
                     comp_variance[[i]] + theme(axis.text.y = element_blank(),axis.ticks.y = element_blank(),axis.title.y = element_blank(), legend.position = 'none' ), 
                         align = 'h', nrow=1, widths = c(4,1))
    # Annotate Figure
    combination1_ann = annotate_figure(
      combination1,
      right = legend
    )
    
    pdf(paste0('figures/03_figures/', figure_name,  mofa_configs$mofa_result[i],  '.pdf'), width =width_par, height =height_par)
    print(combination1_ann)
    dev.off()
    #print(combination1_ann)
    
    }
popup_function_pos(paste0('Generated:', ' figures/03_figures/ ', figure_name,  mofa_configs$mofa_result[i],  '.pdf'))  

In [ ]:
## Save view colors for further usage

In [ ]:
write.csv(data.frame(color_code = unlist(type_colors)),
          paste0('configurations/03_Type_Color_Codes.csv'))

In [ ]:
## Inform that execution was finished
popup_function_pos('03_Run_MOFA: Execution Finished')

In [ ]:
Sys.sleep(20)
popup_function_info('03_Run_MOFA')

In [ ]:
# recreate plot with different order


In [ ]:
library(ggplot2)
library(reshape2)
library(ggpubr)
library(tibble)
library(dplyr)

In [ ]:

res_dir     <- "/mnt/custom-file-systems/efs/fs-0d3b03a3638d516f0_fsap-05c6d9e300aefa0e9/projects/mofa/mofa_workflow/results_mosaic/03_results"
model_id    <- "v31_norecurr_bulkhvg_MOFA"
figure_pref <- "FIG03_Overview_Variance_Decomposition_paper"

color_palette <- c(
  'OPC'             = '#1f77b4',
  'OPC.like'        = '#4b9cd3',
  'Oligodendrocyte' = '#6baed6',
  'Astrocyte'       = '#9ecae1',
  'TRAIL.Astrocytes'= '#ff00ff',
  'AC.like'         = '#6a51a3',
  'NPC.like'        = '#8c6bb1',
  'RG'              = '#bcbddc',
  'Neuron'          = '#dadaeb',
  'TAM.MG'          = '#d62728',
  'TAM.BDM'         = '#e66767',
  'E.MDSCs'         = '#8B4513',
  'M.MDSCs'         = '#DAA520',
  'Mono'            = '#fdae6b',
  'Neutrophil'      = '#fdd0a2',
  'DC'              = '#e31a1c',
  'CD4.CD8'         = '#fc4e2a',
  'NK'              = '#feb24c',
  'B.cell'          = '#ffeda0',
  'Plasma.B'        = '#f768a1',
  'Mast'            = '#ae017e',
  'Endothelial'     = '#31a354',
  'Mural.cell'      = '#74c476',
  'MES.like'        = '#bae4b3',
  # Adding colors for the special views
  'bulk_5000hvg_counts_symbols'    = '#333333', 
  'celltype_proportions_zComp_CLR' = '#999999'
)


csv_file <- file.path(res_dir, paste0("03_MOFA_Variance_Decomposition_", model_id, ".csv"))
mat <- read.csv(csv_file, row.names = 1, check.names = FALSE, stringsAsFactors = FALSE)

data <- melt(as.matrix(mat), varnames = c("Var1","Var2"), value.name = "value")


tot_df <- data.frame(
  Var2 = colnames(mat),
  total_variance = colSums(mat),
  stringsAsFactors = FALSE
)
data <- merge(data, tot_df, by = "Var2")

# Grouped logical order based on categories
view_order <- c(
  # 1. Glial / Neural-related
  "OPC", "Oligodendrocyte", "Astrocyte","Neuron",
  
  # 2. Immune Cells
  "TAM.MG", "TAM.BDM", "E.MDSCs", "M.MDSCs", "Mono", "DC", "CD4.CD8", "NK", "Mast",
  
  # 3. Vascular / Other
  "Endothelial", "Mural.cell",

  # Tumor cells  
    "OPC.like", "AC.like", "NPC.like", "MES.like",
  
  # 4. Special bulk/proportion views (Last)
  "celltype_proportions_zComp_CLR", 
  "bulk_5000hvg_counts_symbols"
)

# Apply factor levels (reversed so the top of the list appears at the top of the plot)
data$Var2 <- factor(data$Var2, levels = rev(view_order))

# Renaming mapping for labels
rename_map <- c(
  "bulk_5000hvg_counts_symbols"    = "RNAseq",
  "celltype_proportions_zComp_CLR" = "Cell Type Proportions"
)

# Helper function for label cleaning
clean_view_names <- function(x) {
  x <- as.character(x)
  if (x %in% names(rename_map)) {
    return(rename_map[x])
  } else {
    # Replace dots with hyphens (e.g., TAM.MG -> TAM-MG)
    return(gsub("\\.", "-", x))
  }
}

# Create the labeled factor for the Y-axis
data$Var2_labeled <- sapply(data$Var2, clean_view_names)
labeled_order <- sapply(rev(view_order), clean_view_names)
data$Var2_labeled <- factor(data$Var2_labeled, levels = labeled_order)


xlabel <- xlab("Factor")
ylabel <- ylab("View")

# Heatmap
p_heat <- ggplot(data, aes(x = Var1, y = Var2_labeled, fill = value)) +
  geom_tile() +
  scale_fill_gradient(low = "white", high = "black", name = NULL) + # Removed legend title
  xlabel + ylabel +
  theme_minimal() + 
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1),
        legend.position = "right")

# Bar-chart
plot_complete <- unique(data[, c("Var2", "Var2_labeled", "total_variance")])
p_bar <- ggplot(plot_complete, aes(x = Var2_labeled, y = total_variance, fill = Var2)) +
  geom_bar(stat = "identity") +
  coord_flip() +
  xlabel + 
  ylab("Explained Variance") + # This will appear on the horizontal axis due to coord_flip
  theme_minimal() + 
  scale_fill_manual(values = color_palette) + 
  theme(
    legend.position = "none",
    panel.grid.minor = element_blank(),
    panel.grid.major.y = element_blank()
  )


legend       <- get_legend(p_heat)
p_heat_noleg <- p_heat + theme(legend.position = "none")

combo <- ggarrange(
  p_heat_noleg,
  p_bar + theme(axis.text.y = element_blank(), 
                axis.ticks.y = element_blank(), 
                axis.title.y = element_blank()),
  nrow = 1, widths = c(4, 1), align = "h"
)
combo_annot <- annotate_figure(combo, right = legend)

out_pdf <- file.path("figures/03_figures", paste0(figure_pref, model_id, ".pdf"))
dir.create(dirname(out_pdf), recursive = TRUE, showWarnings = FALSE)

pdf(out_pdf, width = 10, height = 6)
print(combo_annot)
dev.off()

message("Recreated figure at: ", out_pdf)